In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import missingno as msno
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("/content/dataset.csv", encoding='utf-8')

In [ ]:
df = df[~(df["Exam_Score"] > 100)]

In [ ]:
mode = df["Teacher_Quality"].mode()[0]
df["Teacher_Quality"] = df["Teacher_Quality"].fillna(mode)

In [ ]:
mode_pv = df["Parental_Education_Level"].mode()[0]
df["Parental_Education_Level"] = df["Parental_Education_Level"].fillna(mode_pv)

In [ ]:
mode_pv = df["Distance_from_Home"].mode()[0]
df["Distance_from_Home"] = df["Distance_from_Home"].fillna(mode_pv)

In [ ]:
# iqr outliers
q1 = df["Exam_Score"].quantile(0.25)
q3 = df["Exam_Score"].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
iqr_outlier_mask = (
    (df["Exam_Score"] < lower_bound) |
    (df["Exam_Score"] > upper_bound)
)
df = df[~iqr_outlier_mask]

In [ ]:
iqr_outlier_pct = iqr_outlier_mask.mean() * 100

print(f"iqr outliers: {iqr_outlier_pct:.2f}%")

iqr outliers: 1.56%


In [ ]:
# z-index outliers:
mean_score = df["Exam_Score"].mean()
std_score = df["Exam_Score"].std()

df["Exam_Score_zscore"] = (
    (df["Exam_Score"] - mean_score) / std_score
)
df = df[~(df["Exam_Score_zscore"] > 3)]
df.drop("Exam_Score_zscore",axis=1,inplace=True)

binary encoding for Extracurricular_Activities

One-hot encoding des variables nominales (Gender, SchoolType, ExtracurricularActivities, InternetAccess, LearningDisabilities, Peer_Influence)


In [ ]:
df['Extracurricular_Activities'] = (df["Extracurricular_Activities"] == "Yes").astype(int)
df["Internet_Access"] = (df["Internet_Access"]== "Yes").astype(int)
df["Learning_Disabilities"] = (df["Learning_Disabilities"] == "Yes").astype(int)

use **Ordinal encoding** on the Peer influence values

Use **OneHotEncoder** on "School_Type","Gender" becuse they are a nominal columns and

**Encodage ordinal** des variables ordinales (ParentalInvolvement, AccesstoResources, MotivationLevel, FamilyIncome, TeacherQuality, ParentalEducationLevel, DistancefromHome).

In [ ]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
import pandas as pd

ordinal_columns = [
    "Parental_Involvement",
    "Access_to_Resources",
    "Motivation_Level",
    "Family_Income",
    "Teacher_Quality",
    "Parental_Education_Level",
    "Distance_from_Home",
]

ordinal_categories = [
    ["Low", "Medium", "High"],
    ["Low", "Medium", "High"],
    ["Low", "Medium", "High"],
    ["Low", "Medium", "High"],
    ["Low", "Medium", "High"],
    ["High School", "College", "Postgraduate"],
    ["Near", "Moderate", "Far"],
]

encoder = OrdinalEncoder(categories=ordinal_categories)
df[ordinal_columns] = encoder.fit_transform(df[ordinal_columns])

nominal_columns = ["School_Type", "Gender", "Peer_Influence"]

onehot = OneHotEncoder(
    drop="if_binary",
    handle_unknown="ignore",
    sparse_output=False
)

encoded = onehot.fit_transform(df[nominal_columns])

encoded_df = pd.DataFrame(
    encoded,
    columns=onehot.get_feature_names_out(nominal_columns),
    index=df.index
)

df = pd.concat([df.drop(columns=nominal_columns), encoded_df], axis=1)

replace the old column with the coded ones

standard the numeric columns using **SimpleImputer** + **StandardScaler** form **Scikit-learn**


In [ ]:
df[ordinal_columns]

,Parental_Involvement,Access_to_Resources,Motivation_Level,Family_Income,Teacher_Quality,Parental_Education_Level,Distance_from_Home
0,0.0,2.0,0.0,0.0,1.0,0.0,0.0
1,0.0,1.0,0.0,1.0,1.0,1.0,1.0
2,1.0,1.0,1.0,1.0,1.0,2.0,0.0
3,0.0,1.0,1.0,1.0,1.0,0.0,1.0
4,1.0,1.0,1.0,1.0,2.0,1.0,0.0
...,...,...,...,...,...,...,...
6602,2.0,1.0,1.0,2.0,1.0,0.0,0.0
6603,2.0,1.0,1.0,0.0,2.0,0.0,0.0
6604,1.0,0.0,0.0,0.0,1.0,2.0,0.0
6605,2.0,2.0,2.0,0.0,1.0,0.0,2.0


In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Exam_Score"])
y = df["Exam_Score"]
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=None
)

prepare a pipline to replace the invalide numbers with the median and standard the values

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_columns = [
    "Hours_Studied",
    "Attendance",
    "Sleep_Hours",
    "Previous_Scores",
    "Tutoring_Sessions",
    "Physical_Activity",
]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

use the standard nuemric pipline on splites data

In [ ]:
X_train_numeric = numeric_transformer.fit_transform(
    X_train[numeric_columns]
)

X_test_numeric = numeric_transformer.transform(
    X_test[numeric_columns]
)

Processor of data in more efecent way i will implement it later

In [ ]:
# from sklearn.compose import ColumnTransformer
# from sklearn.preprocessing import OneHotEncoder

# preprocessor = ColumnTransformer(
#     transformers=[
#         ("num", numeric_transformer, numeric_columns),

#         ("nominal", OneHotEncoder(
#             handle_unknown="ignore"
#         ), nominal_columns),

#         ("ordinal", ordinal_encoder, ordinal_columns)
#     ],
#     verbose_feature_names_out=False
# )

In [ ]:
from sklearn.linear_model import Ridge
model = Ridge(alpha=1.0)
model.fit(X_train,y_train)

Ridge()

In [ ]:
Y_pred = model.predict(X_test)

add those: greedsearch | kfalde

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score
)

mae = mean_absolute_error(y_test, Y_pred)
rmse = root_mean_squared_error(y_test, Y_pred)
r2 = r2_score(y_test, Y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

MAE: 0.2715872465200424
RMSE: 0.32707255573327193
R²: 0.9897217335837031


test the model

In [ ]:
predictions_array = model.predict(X_test)
pdf = pd.DataFrame(predictions_array, index=y_test.index, columns=['Predicted_Exam_Score'])
tdf = y_test.rename('True_Exam_Score')
newdf = pd.concat([pdf, tdf], axis=1)

,Predicted_Exam_Score,True_Exam_Score
5491,68.366719,68
268,64.694212,65
6119,68.655001,69
5306,73.533019,74
2569,67.778177,68
...,...,...
2392,67.378664,67
2112,67.651494,68
1490,64.135888,64
2566,64.073229,64
